# Plots FedMAD (adaptado de plots.ipynb)

Adaptacao do notebook original do PFLlib para os resultados do sistema MAD.
Estrategias de visualizacao:

1. **Seguranca (BA x ASR)** - acuracia benigna e taxa de sucesso do ataque label por rodada, comparando FedAvg vs FedMAD;
2. **Espaco latente (t-SNE)** - embeddings dos modelos dos clientes (benignos x maliciosos) e trajetoria do modelo global (encoder SimCLR);
3. **Robustez** - ASR/BA finais conforme a porcentagem de clientes maliciosos;
4. **Eficiencia** - latencia de deteccao por agente/agregador (ms) e overhead do round;
5. **Distribuicao de scores** - gap entre benignos (baixo) e maliciosos (alto) no score final, justificando o limiar de remocao (gamma).

**Fonte de dados:** `*_agentlog.json` (escrito por `servermad.save_agent_results()`)
e `.h5` (baselines FedAvg / train_loss). Requer scikit-learn para o t-SNE.

In [ ]:
import json
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = 'results/'
PLOTS_DIR = os.path.join(RESULTS_DIR, 'plots')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

In [ ]:
agentlog_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*_agentlog.json')))
h5_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.h5')))

print(f'Arquivos agentlog (FedMAD) ({len(agentlog_files)}):')
for f in agentlog_files:
    print(f'  {os.path.basename(f)}')

print(f'\nArquivos H5 (PFLlib, para train_loss e baselines) ({len(h5_files)}):')
for f in h5_files:
    print(f'  {os.path.basename(f)}')

In [ ]:
def parse_config(stem):
    """Configuracao extraida do nome: dataset_algo_cc_rfake_nmal_[atk]_goal_times."""
    parts = stem.split('_')
    cfg = {
        'dataset': parts[0],
        'algorithm': parts[1],
        'cc': parts[2],
        'rate_fake': parts[3] if len(parts) > 3 else '?',
        'nmal': parts[4] if len(parts) > 4 else '?',
        'atk': '-',
        'goal_times': '_'.join(parts[5:]),
    }
    if len(parts) > 5 and parts[5] in ('all', 'label', 'random', 'zero', 'shuffle'):
        cfg['atk'] = parts[5]
        cfg['goal_times'] = '_'.join(parts[6:])
    return cfg


def load_h5(path):
    import h5py
    data = {'_file': os.path.basename(path)}
    stem = data['_file'][:-len('.h5')]
    data['_config'] = parse_config(stem)
    with h5py.File(path, 'r') as hf:
        for k in ('rs_test_acc', 'rs_test_auc', 'rs_train_loss', 'rs_asr', 'rs_ba', 'rs_train_time'):
            if k in hf:
                data[k] = np.array(hf[k])
    data['_source'] = 'h5'
    return data


def load_agentlog(path):
    with open(path) as f:
        data = json.load(f)
    data['_file'] = os.path.basename(path)
    stem = data['_file'][:-len('_agentlog.json')]
    data['_config'] = parse_config(stem)
    data['_source'] = 'agentlog'
    data.setdefault('rs_asr', np.array([]))
    data.setdefault('rs_ba', np.array([]))

    # train_loss nao esta no agentlog: tenta casar com o .h5 correspondente
    data['rs_train_loss'] = np.array([])
    h5 = os.path.join(os.path.dirname(path) or '.', stem + '.h5')
    if os.path.exists(h5):
        try:
            import h5py
            with h5py.File(h5, 'r') as hf:
                if 'rs_train_loss' in hf:
                    data['rs_train_loss'] = np.array(hf['rs_train_loss'])
        except Exception:
            pass
    return data


def load_run(path):
    """Carrega um experimento, seja agentlog (MAD) ou h5 (FedAvg/baseline)."""
    if path.endswith('_agentlog.json'):
        return load_agentlog(path)
    return load_h5(path)


def all_run_files():
    """Todos os experimentos, sem duplicar MAD (agentlog) com seu proprio h5."""
    agent_base = {os.path.basename(f)[:-len('_agentlog.json')] for f in agentlog_files}
    runs = [f for f in h5_files if os.path.basename(f)[:-len('.h5')] not in agent_base]
    return sorted(runs + agentlog_files)


def final_asr_ba(data):
    a = np.asarray(data.get('rs_asr', []), dtype=float)
    b = np.asarray(data.get('rs_ba', []), dtype=float)
    if len(a) and len(b):
        return a[-1], b[-1]
    return None, None


def detection_metrics(data):
    """Removidos e TP/FN por round a partir do ground truth."""
    mali = set(data['malicious_indices'])
    n_mal = len(mali)
    rounds, removed, tp, fn = [], [], [], []
    for e in data['agent_round_log']:
        rem = set(e['removed_ids'])
        t = len(rem & mali)
        n = n_mal - t if n_mal else 0
        rounds.append(e['round'])
        removed.append(len(rem))
        tp.append(t)
        fn.append(n)
    return (np.array(rounds), np.array(removed), np.array(tp), np.array(fn))


def detection_f1(tp, fn, removed):
    prec = np.where(removed > 0, tp / np.maximum(removed, 1), 0.0)
    den = tp + fn
    rec = np.where(den > 0, tp / np.maximum(den, 1), 0.0)
    return np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec), 0.0)


def score_auc(s, y):
    """AUC de um vetor de scores vs labels binarias (sem sklearn)."""
    if len(np.unique(y)) < 2:
        return np.nan
    order = np.argsort(s, kind='mergesort')
    s_sorted = s[order]
    ranks = np.empty(len(s), dtype=float)
    i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s_sorted[j + 1] == s_sorted[i]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2.0 + 1
        i = j + 1
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan
    return (ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def agent_effectiveness(data, threshold=0.6):
    """Eficacia de defesa por agente (e do score final do servidor).

    Usa o MESMO score que o servidor usa na agregacao: o agente gera um score,
    o servidor remove clientes com score > threshold (0.6). Para cada agente e
    para o score final agregado, calcula por round:
      - recall@threshold: fracoes dos maliciosos que o score pegaria
      - precision@threshold: dos removidos, quantos sao maliciosos
      - auc: qualidade de separacao malicioso x benigno
    """
    mali = set(data['malicious_indices'])
    agent_names = list(data['agent_names']) + ['Final']
    rounds = []
    eff = {name: {'recall': [], 'precision': [], 'auc': []} for name in agent_names}
    for e in data['agent_round_log']:
        rounds.append(e['round'])
        ids = [int(c) for c in e['client_ids_uploaded']]
        y = np.array([1 if c in mali else 0 for c in ids])
        for name in agent_names:
            if name == 'Final':
                sdict = e['final_scores']
            else:
                sdict = e['agent_scores'].get(name, {})
            s = np.array([sdict.get(str(c), np.nan) for c in ids])
            mask = ~np.isnan(s)
            if mask.sum() < 2:
                eff[name]['recall'].append(np.nan)
                eff[name]['precision'].append(np.nan)
                eff[name]['auc'].append(np.nan)
                continue
            ss, yy = s[mask], y[mask]
            pred = ss > threshold
            tp = int((pred & (yy == 1)).sum())
            fp = int((pred & (yy == 0)).sum())
            fn = int(((~pred) & (yy == 1)).sum())
            eff[name]['recall'].append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
            eff[name]['precision'].append(tp / (tp + fp) if (tp + fp) > 0 else 0.0)
            eff[name]['auc'].append(score_auc(ss, yy))
    return np.array(rounds), eff

In [ ]:
AGENT_INDEX = -1

if agentlog_files:
    data = load_agentlog(agentlog_files[AGENT_INDEX])
    cfg = data['_config']
    print(f'Arquivo: {data["_file"]}\n')

    print('--- Configuracao do Experimento ---')
    for k, v in cfg.items():
        print(f'  {k}: {v}')
    print(f'  clientes: {data["num_clients"]}')
    print(f'  rounds globais: {data["global_rounds"]}')
    print(f'  maliciosos: {data["n_client_malicious"]} -> {data["malicious_indices"]}')
    print(f'  agentes: {", ".join(data["agent_names"])}')

    print('\n--- Desempenho (melhores) ---')
    if len(data['rs_test_acc']):
        print(f'  test_acc best: {max(data["rs_test_acc"]):.4f}')
    if len(data['rs_test_auc']):
        print(f'  test_auc best: {max(data["rs_test_auc"]):.4f}')
    if len(data['rs_asr']) and len(data['rs_ba']):
        print(f'  ASR final: {data["rs_asr"][-1]:.4f}')
        print(f'  BA final : {data["rs_ba"][-1]:.4f}')

    print('\n--- Deteccao por round ---')
    rounds, removed, tp, fn = detection_metrics(data)
    for r, nrem, t in zip(rounds, removed, tp):
        print(f'  round {int(r):3d}: removidos={int(nrem):2d}  acertou_malicioso={int(t)}/{len(set(data["malicious_indices"]))}')
else:
    print('Nenhum arquivo agentlog encontrado.')

In [ ]:
if agentlog_files:
    data = load_agentlog(agentlog_files[AGENT_INDEX])
    cfg = data['_config']
    test_acc = np.array(data['rs_test_acc'])
    test_auc = np.array(data['rs_test_auc'])
    train_loss = np.array(data['rs_train_loss'])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    title = f'{cfg["algorithm"]} | cc={cfg["cc"]} | nmal={cfg["nmal"]} | atk={cfg["atk"]}'

    # Accuracy
    if len(test_acc) > 0:
        axes[0].plot(test_acc, color='#2196F3', linewidth=1.5)
        axes[0].set_title('Test Accuracy')
        axes[0].set_xlabel('Evaluation Round')
        axes[0].set_ylabel('Accuracy')
        best = test_acc.max()
        axes[0].axhline(y=best, color='red', linestyle='--', alpha=0.5, label=f'Best: {best:.4f}')
        axes[0].legend()

    # AUC
    if len(test_auc) > 0:
        axes[1].plot(test_auc, color='#4CAF50', linewidth=1.5)
        axes[1].set_title('Test AUC')
        axes[1].set_xlabel('Evaluation Round')
        axes[1].set_ylabel('AUC')
        best_auc = test_auc.max()
        axes[1].axhline(y=best_auc, color='red', linestyle='--', alpha=0.5, label=f'Best: {best_auc:.4f}')
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'AUC nao disponivel', ha='center', va='center', transform=axes[1].transAxes)

    # Loss (requer o .h5)
    if len(train_loss) > 0:
        axes[2].plot(train_loss, color='#FF5722', linewidth=1.5)
        axes[2].set_title('Train Loss')
        axes[2].set_xlabel('Evaluation Round')
        axes[2].set_ylabel('Loss')
        min_loss = train_loss.min()
        axes[2].axhline(y=min_loss, color='blue', linestyle='--', alpha=0.5, label=f'Min: {min_loss:.4f}')
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, 'Train Loss nao disponivel\n(requer o .h5)', ha='center', va='center', transform=axes[2].transAxes)

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Nenhum arquivo agentlog encontrado.')

In [ ]:
if agentlog_files:
    data = load_agentlog(agentlog_files[AGENT_INDEX])
    cfg = data['_config']
    rounds, eff = agent_effectiveness(data, threshold=0.6)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    colors = plt.cm.tab10.colors

    # Eficacia de cada agente + score final (o que o servidor usa na agregacao)
    for i, name in enumerate(data['agent_names']):
        axes[0].plot(rounds, eff[name]['recall'], 'o-', markersize=3, linewidth=1.2,
                     color=colors[i], label=name, alpha=0.7)
    axes[0].plot(rounds, eff['Final']['recall'], 'k-', linewidth=2, label='Final (server)', alpha=0.9)
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].set_title('Recall@0.6 por Rodada (mesma regra de remocao do servidor)')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Recall')
    axes[0].legend(fontsize=9)

    for i, name in enumerate(data['agent_names']):
        axes[1].plot(rounds, eff[name]['auc'], 'o-', markersize=3, linewidth=1.2,
                     color=colors[i], label=name, alpha=0.7)
    axes[1].plot(rounds, eff['Final']['auc'], 'k-', linewidth=2, label='Final (server)', alpha=0.9)
    axes[1].set_ylim(0.3, 1.05)
    axes[1].set_title('AUC por Rodada (separacao malicioso x benigno)')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('AUC')
    axes[1].legend(fontsize=9)

    plt.suptitle(f'Eficacia de Defesa dos Agentes | {cfg["algorithm"]} cc={cfg["cc"]} nmal={cfg["nmal"]}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Nenhum arquivo agentlog encontrado.')

In [ ]:
# ---------------------------------------------------------------
# Estrategia 1 - Seguranca: BA (Benign Accuracy) x ASR (Attack Success
# Rate) por rodada para todos os experimentos com ataque "label".
# FedMAD (linhas solidas) deve manter o ASR baixo vs FedAvg (tracejado).
# ---------------------------------------------------------------
run_files = all_run_files()
label_runs = []
for p in run_files:
    d = load_run(p)
    if d['_config']['atk'] != 'label':
        continue
    a = np.asarray(d.get('rs_asr', []), dtype=float)
    b = np.asarray(d.get('rs_ba', []), dtype=float)
    if len(a) == 0 or len(b) == 0:
        continue
    label_runs.append(d)

if label_runs:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for d in label_runs:
        cfg = d['_config']
        x = np.arange(len(d['rs_asr']))
        mal = int(cfg['nmal'])
        is_mad = cfg['algorithm'] == 'MAD'
        style = '-' if is_mad else '--'
        axes[0].plot(x, d['rs_asr'], style, linewidth=1.5,
                     label=f"{cfg['algorithm']} nmal={mal}")
        axes[1].plot(x, d['rs_ba'], style, linewidth=1.5,
                     label=f"{cfg['algorithm']} nmal={mal}")
    axes[0].set_title('ASR (retencao do label flip) - quanto menor, melhor')
    axes[0].set_xlabel('Evaluation Round')
    axes[0].set_ylabel('ASR')
    axes[0].set_ylim(-0.02, 1.02)
    axes[0].legend(fontsize=9)
    axes[1].set_title('BA (acuracia sobre clientes benignos)')
    axes[1].set_xlabel('Evaluation Round')
    axes[1].set_ylabel('BA')
    axes[1].set_ylim(-0.02, 1.02)
    axes[1].legend(fontsize=9)
    plt.suptitle('Seguranca: BA x ASR (ataque label)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    os.makedirs(PLOTS_DIR, exist_ok=True)
    plt.savefig(os.path.join(PLOTS_DIR, 'ba_asr.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print('Plot salvo: results/plots/ba_asr.png')
else:
    print('Nenhum experimento "label" com ASR/BA salvo. Rode: bash run_fedmad_test.sh attack_label')

In [ ]:
# ---------------------------------------------------------------
# Estrategia 2 - Espaco latente: t-SNE dos embeddings dos modelos
# dos clientes (benignos x maliciosos) e trajetoria do modelo global.
# ---------------------------------------------------------------
def get_tsne_experiment():
    for p in agentlog_files:
        d = load_agentlog(p)
        embs = [e.get('client_embeddings') for e in d.get('agent_round_log', [])]
        if any(embs):
            return d
    return None

d = get_tsne_experiment()
if d is None:
    print('Nenhum experimento com client_embeddings salvo (requer a instrumentacao t-SNE do servermad).')
else:
    try:
        from sklearn.manifold import TSNE
    except ImportError:
        print('scikit-learn nao instalado. Rode: pip install scikit-learn')
    else:
        mali = set(d['malicious_indices'])
        X, labels, rounds = [], [], []
        for e in d['agent_round_log']:
            embs = e.get('client_embeddings', {})
            r = e['round']
            for cid in e['client_ids_uploaded']:
                v = embs.get(str(cid))
                if v is None:
                    continue
                X.append(v)
                labels.append('mal' if int(cid) in mali else 'ben')
                rounds.append(r)
            g = e.get('global_embedding')
            if g:
                X.append(g)
                labels.append('global')
                rounds.append(r)
        X = np.asarray(X, dtype=float)
        n = len(X)
        perp = min(30.0, max(5.0, (n - 1) / 3))
        Y = TSNE(n_components=2, init='pca', random_state=0, perplexity=perp).fit_transform(X)

        lbl = np.array(labels)
        ben, mal, glo = lbl == 'ben', lbl == 'mal', lbl == 'global'
        fig, ax = plt.subplots(figsize=(9, 7))
        ax.scatter(Y[ben, 0], Y[ben, 1], c='#2196F3', s=14, alpha=0.55, label='benigno')
        ax.scatter(Y[mal, 0], Y[mal, 1], c='#F44336', s=16, marker='x', alpha=0.8, label='malicioso')
        if glo.any():
            g_order = np.argsort(np.array(rounds)[glo])
            ax.plot(Y[glo, 0][g_order], Y[glo, 1][g_order], 'o-', color='black', lw=1.2, ms=3, label='modelo global')
        ax.set_title(f"t-SNE do espaco latente | {d['_config']['algorithm']} nmal={d['_config']['nmal']} atk={d['_config']['atk']}")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        os.makedirs(PLOTS_DIR, exist_ok=True)
        plt.savefig(os.path.join(PLOTS_DIR, 'tsne_latent.png'), dpi=150, bbox_inches='tight')
        plt.close()
        print('Plot salvo: results/plots/tsne_latent.png')

In [ ]:
# ---------------------------------------------------------------
# Estrategia 3 - Robustez: ASR/BA finais x porcentagem de clientes
# maliciosos (serie -atk label com nmc 3/5/8/10).
# ---------------------------------------------------------------
rob = {}
for p in all_run_files():
    d = load_run(p)
    if d['_config']['atk'] != 'label':
        continue
    a, b = final_asr_ba(d)
    if a is None:
        continue
    try:
        nmal = int(d['_config']['nmal'])
    except (TypeError, ValueError):
        continue
    nclients = d.get('num_clients') or 50
    pct = nmal / nclients * 100
    rob.setdefault(d['_config']['algorithm'], []).append((pct, a, b))

if len(rob) and max(len(v) for v in rob.values()) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for algo, rows in rob.items():
        rows.sort()
        xs = [r[0] for r in rows]
        axes[0].plot(xs, [r[1] for r in rows], 'o-', linewidth=2, label=algo)
        axes[1].plot(xs, [r[2] for r in rows], 'o-', linewidth=2, label=algo)
    axes[0].set_title('ASR final x % maliciosos')
    axes[0].set_xlabel('% clientes maliciosos')
    axes[0].set_ylabel('ASR final')
    axes[0].set_ylim(-0.02, 1.02)
    axes[0].legend()
    axes[1].set_title('BA final x % maliciosos')
    axes[1].set_xlabel('% clientes maliciosos')
    axes[1].set_ylabel('BA final')
    axes[1].set_ylim(-0.02, 1.02)
    axes[1].legend()
    plt.suptitle('Robustez: ASR deve permanecer baixo conforme a cota de maliciosos cresce', fontsize=14, fontweight='bold')
    plt.tight_layout()
    os.makedirs(PLOTS_DIR, exist_ok=True)
    plt.savefig(os.path.join(PLOTS_DIR, 'robustness_asr.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print('Plot salvo: results/plots/robustness_asr.png')
else:
    print('Sao necessarios >= 2 experimentos "label" do mesmo algoritmo (nmc 3/5/8/10). Rode: bash run_fedmad_test.sh robustness')

In [ ]:
# ---------------------------------------------------------------
# Estrategia 4 - Eficiencia: latencia de deteccao por agente e do
# agregador (ms/round) + overhead de deteccao em relacao ao round.
# ---------------------------------------------------------------
def get_latency_experiment():
    for p in agentlog_files:
        d = load_agentlog(p)
        if any(e.get('agent_times') for e in d.get('agent_round_log', [])):
            return d
    return None

d = get_latency_experiment()
if d is None:
    print('Nenhum experimento com agent_times salvo (requer a instrumentacao de latencia do servermad).')
else:
    names = list(d['agent_names'])
    per_agent = {n: [] for n in names}
    agg_vals, over_frac = [], []
    budget = d.get('budget', [])
    for e in d['agent_round_log']:
        at = e.get('agent_times', {})
        for n in names:
            if n in at:
                per_agent[n].append(at[n] * 1000.0)
        agg_vals.append(e.get('agg_time', 0.0) * 1000.0)
        det = sum(at.values()) + e.get('agg_time', 0.0)
        r = e['round']
        if r < len(budget) and budget[r] > 0:
            over_frac.append(det / budget[r] * 100.0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    means = [np.mean(v) if v else 0.0 for v in per_agent.values()]
    means.append(np.mean(agg_vals) if agg_vals else 0.0)
    labs = names + ['Agregador']
    axes[0].barh(labs[::-1], means[::-1], color='#7E57C2')
    axes[0].set_title('Latencia media por round (ms) - deteccao')
    axes[0].set_xlabel('ms')
    if over_frac:
        axes[1].plot(np.arange(len(over_frac)), over_frac, 'o-', color='#FF9800')
        axes[1].set_title('Overhead de deteccao (% do tempo do round)')
        axes[1].set_xlabel('Round')
        axes[1].set_ylabel('%')
    else:
        axes[1].text(0.5, 0.5, 'sem budget', ha='center', va='center', transform=axes[1].transAxes)
    plt.suptitle(f"Eficiencia dos agentes | {d['_config']['algorithm']} nmal={d['_config']['nmal']}",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    os.makedirs(PLOTS_DIR, exist_ok=True)
    plt.savefig(os.path.join(PLOTS_DIR, 'latency_agents.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print('Plot salvo: results/plots/latency_agents.png')

In [ ]:
# ---------------------------------------------------------------
# Estrategia 5 - Distribuicao de scores: gap entre benignos (baixo)
# e maliciosos (alto) no score final do servidor -> justifica o limiar.
# ---------------------------------------------------------------
if agentlog_files:
    d = load_agentlog(agentlog_files[AGENT_INDEX])
    mali = set(d['malicious_indices'])
    s_ben, s_mal, r_ben, r_mal = [], [], [], []
    for e in d['agent_round_log']:
        r = e['round']
        for cid in e['client_ids_uploaded']:
            v = e['final_scores'].get(str(cid))
            if v is None:
                continue
            if int(cid) in mali:
                s_mal.append(v); r_mal.append(r)
            else:
                s_ben.append(v); r_ben.append(r)
    if s_mal and s_ben:
        s_ben, s_mal = np.array(s_ben), np.array(s_mal)
        gap = s_mal.mean() - s_ben.mean()
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].hist(s_ben, bins=30, alpha=0.6, color='#2196F3', label=f'benigno (n={len(s_ben)})')
        axes[0].hist(s_mal, bins=30, alpha=0.6, color='#F44336', label=f'malicioso (n={len(s_mal)})')
        axes[0].axvline(0.6, color='black', ls='--', lw=1.5, label='gamma = 0.6')
        axes[0].set_title('Score final (server) - benigno x malicioso')
        axes[0].set_xlabel('score')
        axes[0].legend(fontsize=9)
        axes[1].scatter(r_mal, s_mal, c='#F44336', s=12, alpha=0.6, label='malicioso')
        axes[1].scatter(r_ben, s_ben, c='#2196F3', s=8, alpha=0.4, label='benigno')
        axes[1].axhline(0.6, color='black', ls='--', lw=1.2)
        axes[1].set_title('Score final por round (limiar de remocao = 0.6)')
        axes[1].set_xlabel('Round')
        axes[1].set_ylabel('score')
        axes[1].legend(fontsize=9)
        plt.suptitle(f'Distribuicao de scores - gap malicioso-benigno = {gap:.3f} (justifica o limiar gamma)',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        os.makedirs(PLOTS_DIR, exist_ok=True)
        plt.savefig(os.path.join(PLOTS_DIR, 'score_gap.png'), dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Plot salvo: results/plots/score_gap.png (gap = {gap:.3f})')
    else:
        print('Sem dados de score final no experimento selecionado.')
else:
    print('Nenhum arquivo agentlog encontrado.')

In [ ]:
# Selecione os indices dos arquivos agentlog para comparar
# Exemplo: COMPARE_INDICES = [0, 1, 2] para comparar os 3 primeiros
COMPARE_INDICES = list(range(len(agentlog_files)))  # todos por padrao

if len(agentlog_files) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for idx in COMPARE_INDICES:
        if idx >= len(agentlog_files):
            continue
        d = load_agentlog(agentlog_files[idx])
        cfg = d['_config']
        acc = np.array(d['rs_test_acc'])
        rounds, removed, tp, fn = detection_metrics(d)
        label = f'{cfg["algorithm"]}_cc{cfg["cc"]}_nmal{cfg["nmal"]}_atk{cfg["atk"]}'

        if len(acc) > 0:
            axes[0].plot(acc, linewidth=1.5, label=label)
        if len(rounds) > 0:
            f1 = detection_f1(tp, fn, removed)
            axes[1].plot(rounds, f1, linewidth=1.5, label=label)

    axes[0].set_title('Test Accuracy')
    axes[0].set_xlabel('Evaluation Round')
    axes[0].legend(fontsize=9)

    axes[1].set_title('F1 de Deteccao por Round')
    axes[1].set_xlabel('Round')
    axes[1].legend(fontsize=9)

    plt.suptitle('Comparacao entre Experimentos', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
elif len(agentlog_files) == 1:
    print('Apenas 1 arquivo agentlog disponivel. Rode mais experimentos para comparar.')
else:
    print('Nenhum arquivo agentlog encontrado.')

In [ ]:
eff_by_cc = {}

for path in agentlog_files:
    d = load_agentlog(path)
    cc = d['_config']['cc']
    rounds, eff = agent_effectiveness(d, threshold=0.6)
    eff_by_cc.setdefault(cc, []).append((rounds, eff))


def mean_by_round(series_list, name, metric):
    """Media de uma metrica por round, agregando os experimentos de um mesmo cc."""
    by_round = {}
    for rounds, eff in series_list:
        arr = eff[name][metric]
        for r, v in zip(rounds, arr):
            by_round.setdefault(int(r), []).append(v)
    rs = sorted(by_round)
    return np.array(rs), np.array([np.nanmean(by_round[r]) for r in rs])


def mean_all(series_list, name, metric):
    vals = [v for rounds, eff in series_list if name in eff for v in eff[name][metric]]
    return float(np.nanmean(vals)) if vals else np.nan


if len(eff_by_cc) >= 2:
    all_names = []
    for cc, series in eff_by_cc.items():
        for rounds, eff in series:
            for n in eff:
                if n not in all_names:
                    all_names.append(n)

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))

    # Eficacia do score final (server) por rodada, agregada por cc
    for cc, series in sorted(eff_by_cc.items()):
        rounds, vals = mean_by_round(series, 'Final', 'auc')
        axes[0].plot(rounds, vals, linewidth=2, label=f'cc={cc}')
    axes[0].set_title('Eficacia do Score Final (server) por Rodada - AUC media por CC')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('AUC')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Eficacia media de cada agente (+ final) por cc (barras)
    x = np.arange(len(all_names))
    n_cc = len(eff_by_cc)
    width = 0.8 / n_cc
    for k, (cc, series) in enumerate(sorted(eff_by_cc.items())):
        means = [mean_all(series, n, 'auc') for n in all_names]
        axes[1].bar(x + (k + 0.5 - n_cc / 2) * width, means, width, label=f'cc={cc}')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(all_names)
    axes[1].set_title('Eficacia Media por Agente (AUC media das rodadas) por CC')
    axes[1].set_xlabel('Agente')
    axes[1].set_ylabel('AUC medio')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle('Eficacia dos Agentes por Configuracao de CC', fontsize=14, fontweight='bold')
    plt.tight_layout()
    os.makedirs(PLOTS_DIR, exist_ok=True)
    plt.savefig(os.path.join(PLOTS_DIR, 'eff_agents_by_cc.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Plot salvo: {os.path.join(PLOTS_DIR, "eff_agents_by_cc.png")}')
else:
    print(f'Dados insuficientes para comparar CCs (encontrados {len(eff_by_cc)} valores distintos).')